In [2]:
import os
import gc
from itertools import zip_longest
from datetime import datetime
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd


"""
ToyNetwork 재분석
- 원본 parquet를 매번 다시 groupby/merge하지 않도록
  sec_time별 processed parquet를 먼저 생성한 뒤 분석에 재사용하는 구조
    최종 수정본
"""


# =============================================================================
# 1. 입력값 설정
# =============================================================================
path = r"C:\Vissim_Workspace\시나리오(140-유고지점 Uniform Random, 검증용)_260610\연결로 영향권_3차로폐쇄\2차\영향권5_진출\3차 수정"

senario_path = r"C:\Users\alswl\Desktop\유고 시나리오\검증용 시나리오\synthetic_scenarios_all_140_with_set_type(영향권5(진출))_3차로폐쇄.csv"

inpx_path = r"C:\Vissim_Workspace\시나리오(140-유고지점 Uniform Random, 검증용)_260610\연결로 영향권_3차로폐쇄\2차\영향권5_진출\화성~서울(140-유고지점)_260526.inpx"

# 한 시나리오당 seed 파일 수
num_mer = 3

# 임계 연속 시간: 초 단위
continuous_list = [180]

# 집계시간: 초 단위
sec_time_list = [10,20,30]
# sec_time_list = [10, 20, 30]

# 상류 검지기 위치
up_dcp_pos_list = [1,2,3]
# up_dcp_pos_list = [1, 2, 3]

start_interval = 1800
end_use_interval = 12600
end_interval = 13200

acc_start_time = 3600
Vc = 53.7

weights = {
    "w1": 1,
    "w2": 1,
    "w3": 1,
    "w4": 1,
    "w5": 1,
    "w6": 1,
}

vehicle_types = [100, 300, 630, 640, 650]

# 램프 관련 검지기
before_ramp = [70, 117, 124, 186, 215]
after_ramp = [74, 121, 127, 190, 221]
input_ramp = [902, 904]
output_ramp = [901, 903, 905]

enter_line = [23, 121, 190]
input_main_ramp = [121, 190]
output_main_ramp = [73, 126, 220]

three_lane = [71, 72, 73, 119, 120, 125, 126, 188, 189, 216, 217, 218, 219, 220]

# 전역 변수: 반복문 내부에서 갱신
revision_time = None
sec_time = None
up_dcp_pos = None
continuous_n = None

lane = None
acc_lane = None
acc_finish_time = None
lane_gradient = None
incident_measurement = None

detector_map = {}


# =============================================================================
# 2. 공통 함수
# =============================================================================
def sort_ascending(df):
    df = df.copy()
    df["StartTime"] = (
        df["TimeGroup"].astype(str).str.split("~").str[0].astype(int)
    )
    df["EndTime"] = (
        df["TimeGroup"].astype(str).str.split("~").str[1].astype(int)
    )
    df = df.sort_values(["StartTime", "EndTime", "New_Measurement"]).reset_index(drop=True)
    return df


def complete_time_measurement(df, value_cols, measurements=None):
    df = df.copy()

    full_time = pd.DataFrame({
        "StartTime": np.arange(start_interval, end_interval, sec_time)
    })
    full_time["EndTime"] = full_time["StartTime"] + sec_time
    full_time["TimeGroup"] = (
        full_time["StartTime"].astype(str) + "~" + full_time["EndTime"].astype(str)
    )

    if measurements is None:
        measurements = sorted(df["New_Measurement"].dropna().unique())

    measurement_df = pd.DataFrame({"New_Measurement": measurements})

    full_index = (
        full_time.assign(key=1)
        .merge(measurement_df.assign(key=1), on="key")
        .drop(columns="key")
    )

    df = full_index.merge(
        df,
        on=["TimeGroup", "New_Measurement"],
        how="left"
    )

    for col in value_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    df = sort_ascending(df)
    return df


def add_time_group(original_df):
    original_df = original_df.copy()

    bins = np.arange(start_interval, end_interval + 1, sec_time)
    labels = [f"{start}~{start + sec_time}" for start in bins[:-1]]

    original_df["TimeGroup"] = pd.cut(
        original_df["t(Entry)"],
        bins=bins,
        labels=labels,
        right=False
    )

    original_df = original_df[original_df["TimeGroup"].notna()].copy()
    original_df["TimeGroup"] = original_df["TimeGroup"].astype(str)

    return original_df


# =============================================================================
# 3. 지표 계산 함수
# =============================================================================
def speed_mean(original_df):
    copy_df = original_df.copy()
    copy_df = copy_df[~copy_df["New_Measurement"].between(900, 910)]

    measurements = sorted(copy_df["New_Measurement"].dropna().unique())

    speed_mean_df = (
        copy_df.groupby(["TimeGroup", "New_Measurement"])
        .agg(
            V_mean=("v[km/h]", "mean"),
            V_count=("v[km/h]", "count")
        )
        .reset_index()
    )

    speed_mean_df = complete_time_measurement(
        speed_mean_df,
        value_cols=["V_mean", "V_count"],
        measurements=measurements
    )

    # 다음 검지기 속도
    speed_mean_df["V_next"] = speed_mean_df.groupby("TimeGroup")["V_mean"].shift(-1)

    cols = ["V_mean", "V_next"]
    speed_mean_df[cols] = speed_mean_df[cols].fillna(0)

    speed_mean_df["delta_V"] = np.where(
        speed_mean_df["V_mean"] == 0,
        0,
        (speed_mean_df["V_next"] - speed_mean_df["V_mean"]) / speed_mean_df["V_mean"]
    )

    return speed_mean_df


def density_mean(speed_df):
    density_mean_df = speed_df.copy()

    density_mean_df["K"] = np.where(
        density_mean_df["V_mean"] == 0,
        0,
        density_mean_df["V_count"] * revision_time / density_mean_df["V_mean"]
    )

    density_mean_df["K_next"] = density_mean_df.groupby("TimeGroup")["K"].shift(-1)

    cols = ["K", "K_next"]
    density_mean_df[cols] = density_mean_df[cols].fillna(0)

    density_mean_df["delta_K"] = np.where(
        density_mean_df["K"] == 0,
        0,
        (density_mean_df["K_next"] - density_mean_df["K"]) / density_mean_df["K"]
    )

    density_mean_df = complete_time_measurement(
        density_mean_df,
        value_cols=["V_mean", "V_count", "K", "K_next", "delta_K"]
    )

    return density_mean_df


def heavy_rate(original_df):
    copy_df = original_df.copy()
    measurements = sorted(copy_df["New_Measurement"].dropna().unique())

    heavy_df = (
        copy_df.groupby(["TimeGroup", "New_Measurement"])
        .agg(
            heavy_count=("Vehicle type", lambda x: x.isin([630, 640, 650]).sum()),
            total_count=("Vehicle type", "count")
        )
        .reset_index()
    )

    heavy_df = sort_ascending(heavy_df)

    heavy_df = complete_time_measurement(
        heavy_df,
        value_cols=["heavy_count", "total_count"],
        measurements=measurements
    )

    heavy_df["rate"] = np.where(
        heavy_df["total_count"] == 0,
        0,
        heavy_df["heavy_count"] / heavy_df["total_count"]
    )

    return heavy_df


def entry_saturation(original_df):
    copy_df = original_df.copy()
    copy_df = copy_df[~copy_df["New_Measurement"].between(900, 910)]

    measurements = sorted(copy_df["New_Measurement"].dropna().unique())

    entry_saturation_df = (
        copy_df.groupby(["TimeGroup", "New_Measurement"])
        .size()
        .reset_index(name="entry_volume")
    )

    entry_saturation_df = complete_time_measurement(
        entry_saturation_df,
        value_cols=["entry_volume"],
        measurements=measurements
    )

    entry_saturation_df = sort_ascending(entry_saturation_df)

    entry_saturation_df["capacity"] = np.where(
        entry_saturation_df["New_Measurement"].isin(three_lane),
        6600,
        4400
    )

    entry_saturation_df["Phi_진입"] = (
        entry_saturation_df["entry_volume"] * revision_time / entry_saturation_df["capacity"]
    )

    return entry_saturation_df


def rfr_rate(original_df):
    copy_df = original_df.copy()
    copy_df = copy_df[copy_df["TimeGroup"].notna()]
    copy_df["TimeGroup"] = copy_df["TimeGroup"].astype(str)

    main_results = []

    for before, after in zip(before_ramp, after_ramp):
        q_before = (
            copy_df[copy_df["New_Measurement"] == before]
            .groupby("TimeGroup")
            .size()
            .reset_index(name="q_before")
        )

        q_after = (
            copy_df[copy_df["New_Measurement"] == after]
            .groupby("TimeGroup")
            .size()
            .reset_index(name="q_after")
        )

        merged = q_before.merge(q_after, on="TimeGroup", how="outer").fillna(0)
        merged["before_ramp"] = before
        merged["after_ramp"] = after
        merged["Qm"] = (merged["q_before"] + merged["q_after"]) / 2
        main_results.append(merged)

    ramp_results = []

    for input_, output_ in zip_longest(input_ramp, output_ramp):
        if output_ is not None:
            q_out = (
                copy_df[copy_df["New_Measurement"] == output_]
                .groupby("TimeGroup")
                .size()
                .reset_index(name="q_out")
            )
            q_out["New_Measurement"] = output_
            ramp_results.append(q_out)

        if input_ is not None:
            q_in = (
                copy_df[copy_df["New_Measurement"] == input_]
                .groupby("TimeGroup")
                .size()
                .reset_index(name="q_in")
            )
            q_in["New_Measurement"] = input_
            ramp_results.append(q_in)

    input_queue = input_main_ramp.copy()
    output_queue = output_main_ramp.copy()
    rfr_list = []

    for i in range(min(len(main_results), len(ramp_results))):
        main_df = main_results[i]
        ramp_df = ramp_results[i]

        rfr_df = pd.merge(main_df, ramp_df, on="TimeGroup", how="outer").fillna(0)

        rfr_df["IR_in"] = 0
        rfr_df["IR_out"] = 0

        if "q_in" in rfr_df.columns:
            rfr_df["IR_in"] = np.where(
                rfr_df["Qm"] == 0,
                0,
                rfr_df["q_in"] / rfr_df["Qm"]
            )
            if input_queue:
                current_input = input_queue.pop(0)
                rfr_df["New_Measurement"] = current_input

        if "q_out" in rfr_df.columns:
            rfr_df["IR_out"] = np.where(
                rfr_df["Qm"] == 0,
                0,
                rfr_df["q_out"] / rfr_df["Qm"]
            )
            if output_queue:
                current_output = output_queue.pop(0)
                rfr_df["New_Measurement"] = current_output

        rfr_df["RFR"] = rfr_df["IR_in"] + rfr_df["IR_out"]
        rfr_list.append(rfr_df)

    if not rfr_list:
        base = copy_df[["TimeGroup"]].drop_duplicates().copy()
        all_measurements = copy_df["New_Measurement"].unique()

        expanded = []
        for m in all_measurements:
            temp = base.copy()
            temp["New_Measurement"] = m
            temp["RFR"] = 0
            expanded.append(temp)

        final_rfr_df = pd.concat(expanded, ignore_index=True)
        final_rfr_df = final_rfr_df.sort_values(["TimeGroup", "New_Measurement"]).reset_index(drop=True)
        final_rfr_df["RFR"] = final_rfr_df["RFR"].fillna(0)

    else:
        final_rfr_df = pd.concat(rfr_list, ignore_index=True)

        target_measurements = input_main_ramp + output_main_ramp
        all_measurements = copy_df["New_Measurement"].unique()

        expanded_df_list = []
        base_rfr_df = final_rfr_df.copy()

        for m in all_measurements:
            if m in target_measurements:
                temp = base_rfr_df[base_rfr_df["New_Measurement"] == m].copy()
            else:
                temp = base_rfr_df[["TimeGroup"]].drop_duplicates().copy()
                temp["New_Measurement"] = m
                temp["RFR"] = 0

            expanded_df_list.append(temp)

        final_rfr_df = pd.concat(expanded_df_list, ignore_index=True)
        final_rfr_df = final_rfr_df.sort_values(["TimeGroup", "New_Measurement"]).reset_index(drop=True)
        final_rfr_df = final_rfr_df[["TimeGroup", "New_Measurement", "RFR"]]
        final_rfr_df["RFR"] = final_rfr_df["RFR"].fillna(0)

    final_rfr_df = sort_ascending(final_rfr_df)

    final_rfr_df = complete_time_measurement(
        final_rfr_df,
        value_cols=["RFR"]
    )

    return final_rfr_df


def output_normality(original_df):
    copy_df = original_df.copy()
    copy_df = copy_df[copy_df["TimeGroup"].notna()]

    normality_list = []

    for enter, exit_ramp, exit_main in zip(enter_line, output_ramp, output_main_ramp):
        entry_df = copy_df[copy_df["New_Measurement"] == enter][
            ["New_Measurement", "VehNo", "t(Entry)"]
        ]

        exit_df = copy_df[copy_df["New_Measurement"] == exit_ramp][
            ["New_Measurement", "VehNo", "t(Entry)"]
        ]

        entry_first = (
            entry_df.groupby("VehNo")["t(Entry)"]
            .min()
            .reset_index()
            .rename(columns={"t(Entry)": "t_entry"})
        )

        exit_first = (
            exit_df.groupby("VehNo")["t(Entry)"]
            .min()
            .reset_index()
            .rename(columns={"t(Entry)": "t_exit"})
        )

        merged = pd.merge(entry_first, exit_first, on="VehNo", how="inner")
        merged["delay_sec"] = merged["t_exit"] - merged["t_entry"]
        merged = merged[merged["delay_sec"] >= 0]

        if len(merged) and np.isfinite(np.nanmedian(merged["delay_sec"])):
            lag_bins = int(round(np.nanmedian(merged["delay_sec"]) / sec_time))
        else:
            lag_bins = 0

        entry_count = (
            copy_df[copy_df["New_Measurement"] == enter]
            .groupby("TimeGroup")
            .size()
            .reset_index(name="Q_in")
        )

        exit_count = (
            copy_df[copy_df["New_Measurement"] == exit_ramp]
            .groupby("TimeGroup")
            .size()
            .reset_index(name="Q_out")
        )

        merged_counts = pd.merge(entry_count, exit_count, on="TimeGroup", how="left")
        merged_counts["Q_out_shift"] = merged_counts["Q_out"].shift(-lag_bins)

        merged_counts["F(outrate)"] = np.where(
            merged_counts["Q_in"] == 0,
            0,
            merged_counts["Q_out_shift"] / merged_counts["Q_in"]
        )

        merged_counts["New_Measurement"] = exit_main
        normality_list.append(merged_counts)

    if not normality_list:
        base = copy_df[["TimeGroup"]].drop_duplicates().copy()
        all_measurements = copy_df["New_Measurement"].unique()

        expanded = []
        for m in all_measurements:
            temp = base.copy()
            temp["New_Measurement"] = m
            temp["F(outrate)"] = 0
            expanded.append(temp)

        final_df = pd.concat(expanded, ignore_index=True)
        final_df = final_df.sort_values(["TimeGroup", "New_Measurement"]).reset_index(drop=True)
        final_df["F(outrate)"] = final_df["F(outrate)"].fillna(0)

    else:
        final_df = pd.concat(normality_list, ignore_index=True)

        all_measurements = copy_df["New_Measurement"].unique()
        expanded_list = []

        for m in all_measurements:
            if m in output_main_ramp:
                temp = final_df[final_df["New_Measurement"] == m].copy()
            else:
                temp = final_df[["TimeGroup"]].drop_duplicates().copy()
                temp["New_Measurement"] = m
                temp["F(outrate)"] = 0

            expanded_list.append(temp)

        final_df = pd.concat(expanded_list, ignore_index=True)
        final_df = final_df.sort_values(["TimeGroup", "New_Measurement"]).reset_index(drop=True)

    final_df = sort_ascending(final_df)

    final_df = complete_time_measurement(
        final_df,
        value_cols=["F(outrate)"]
    )

    return final_df


def calculate_stvm(speed_df, density_df, heavy_df, entry_saturation_df, rfr_df, normality_df):
    merged_df = (
        speed_df[["TimeGroup", "New_Measurement", "delta_V"]]
        .merge(
            density_df[["TimeGroup", "New_Measurement", "delta_K"]],
            on=["TimeGroup", "New_Measurement"],
            how="outer"
        )
        .merge(
            heavy_df[["TimeGroup", "New_Measurement", "rate"]],
            on=["TimeGroup", "New_Measurement"],
            how="outer"
        )
        .merge(
            entry_saturation_df[["TimeGroup", "New_Measurement", "Phi_진입"]],
            on=["TimeGroup", "New_Measurement"],
            how="outer"
        )
        .merge(
            rfr_df[["TimeGroup", "New_Measurement", "RFR"]],
            on=["TimeGroup", "New_Measurement"],
            how="outer"
        )
        .merge(
            normality_df[["TimeGroup", "New_Measurement", "F(outrate)"]],
            on=["TimeGroup", "New_Measurement"],
            how="outer"
        )
    )

    merged_df.fillna(0, inplace=True)

    merged_df["STVM"] = (
        weights["w1"] * merged_df["delta_V"]
        + weights["w2"] * merged_df["delta_K"]
        + weights["w3"] * merged_df["rate"]
        + weights["w4"] * merged_df["Phi_진입"]
        + weights["w5"] * merged_df["RFR"]
        + weights["w6"] * merged_df["F(outrate)"]
    )

    merged_df.replace([np.inf, -np.inf], 0, inplace=True)
    merged_df = modify_frame(merged_df)

    return merged_df


def modify_frame(df):
    modify_df = df.copy()

    if "StartTime" not in modify_df.columns or "EndTime" not in modify_df.columns:
        modify_df["StartTime"] = modify_df["TimeGroup"].str.split("~").str[0].astype(int)
        modify_df["EndTime"] = modify_df["TimeGroup"].str.split("~").str[1].astype(int)

    modify_df = modify_df[
        (modify_df["StartTime"] >= start_interval)
        & (modify_df["EndTime"] <= end_use_interval)
    ]

    modify_df = modify_df[~modify_df["New_Measurement"].isin([266, 901, 902, 903, 904, 905])]
    modify_df = modify_df.sort_values(["StartTime", "New_Measurement"]).reset_index(drop=True)

    return modify_df


# =============================================================================
# 4. processed parquet 생성
# =============================================================================
def build_processed_file(raw_file_path, processed_file_path):
    use_cols = ["Measurem.", "t(Entry)", "v[km/h]", "Vehicle type", "VehNo"]

    df = pd.read_parquet(raw_file_path, columns=use_cols)
    df = df.apply(pd.to_numeric, errors="coerce")

    original_df = df[df["t(Entry)"] != -1.00].reset_index(drop=True)
    original_df.drop(columns=["b[m/s2]", "tQueue", "Occ", "Pers"], inplace=True, errors="ignore")
    original_df["New_Measurement"] = original_df["Measurem."] % 1000

    original_df = add_time_group(original_df)

    speed_df = speed_mean(original_df)
    density_df = density_mean(speed_df)
    heavy_df = heavy_rate(original_df)
    entry_df = entry_saturation(original_df)
    rfr_df = rfr_rate(original_df)
    normality_df = output_normality(original_df)
    stvm_df = calculate_stvm(speed_df, density_df, heavy_df, entry_df, rfr_df, normality_df)

    processed_df = (
        speed_df[[
            "StartTime", "EndTime", "TimeGroup", "New_Measurement",
            "V_mean", "V_count", "V_next", "delta_V"
        ]]
        .merge(
            density_df[["TimeGroup", "New_Measurement", "K", "K_next", "delta_K"]],
            on=["TimeGroup", "New_Measurement"],
            how="left"
        )
        .merge(
            heavy_df[["TimeGroup", "New_Measurement", "rate"]],
            on=["TimeGroup", "New_Measurement"],
            how="left"
        )
        .merge(
            entry_df[["TimeGroup", "New_Measurement", "Phi_진입"]],
            on=["TimeGroup", "New_Measurement"],
            how="left"
        )
        .merge(
            rfr_df[["TimeGroup", "New_Measurement", "RFR"]],
            on=["TimeGroup", "New_Measurement"],
            how="left"
        )
        .merge(
            normality_df[["TimeGroup", "New_Measurement", "F(outrate)"]],
            on=["TimeGroup", "New_Measurement"],
            how="left"
        )
        .merge(
            stvm_df[["TimeGroup", "New_Measurement", "STVM"]],
            on=["TimeGroup", "New_Measurement"],
            how="left"
        )
    )

    processed_df = modify_frame(processed_df)
    processed_df = processed_df.fillna(0)

    processed_df.to_parquet(processed_file_path, index=False)

    del df, original_df
    del speed_df, density_df, heavy_df, entry_df, rfr_df, normality_df, stvm_df, processed_df
    gc.collect()


def build_processed_files(folder_path, parquet_list, processed_folder, overwrite=False):
    os.makedirs(processed_folder, exist_ok=True)

    for mer_file in parquet_list:
        raw_file_path = os.path.join(folder_path, mer_file)
        processed_name = f"processed_{sec_time}sec_{mer_file}"
        processed_file_path = os.path.join(processed_folder, processed_name)

        if os.path.exists(processed_file_path) and not overwrite:
            continue

        print(f"processed 생성: {processed_name}")
        build_processed_file(raw_file_path, processed_file_path)


# =============================================================================
# 5. 분석 함수
# =============================================================================
def get_upstream_detector(root, location):
    detector_no = None
    detector_link = None
    gradient = None

    rsa_lane = None
    rsa_pos = None
    best_pos = -1

    for rsa in root.findall(".//reducedSpeedArea"):
        if rsa.get("name") == location:
            rsa_lane = rsa.get("lane")
            rsa_pos = float(rsa.get("pos"))

            for dcp in root.findall(".//dataCollectionPoint"):
                if dcp.get("lane") == rsa_lane:
                    dcp_pos = float(dcp.get("pos"))

                    if rsa_pos >= dcp_pos >= best_pos:
                        best_pos = dcp_pos
                        detector_no = int(dcp.get("no")) % 1000 - up_dcp_pos
                        detector_link = detector_map.get(detector_no)

    if detector_no is None:
        print("링크 번호가 다름")
        best_pos = float("inf")

        for dcp in root.findall(".//dataCollectionPoint"):
            if dcp.get("lane") == rsa_lane:
                dcp_pos = float(dcp.get("pos"))

                if rsa_pos < dcp_pos < best_pos:
                    best_pos = dcp_pos
                    detector_no = int(dcp.get("no")) % 1000 - 1 - up_dcp_pos
                    detector_link = detector_map.get(detector_no)

    for link in root.findall(".//link"):
        if int(link.get("no")) == detector_link:
            gradient = round(float(link.get("gradient")), 4)
            break

    return detector_no, gradient


def calculate_logD(speed_avg, entry_avg, heavy_avg, stvm_avg):
    speed_df = speed_avg.copy()
    entry_df = entry_avg.copy()
    heavy_df = heavy_avg.copy()
    stvm_df = stvm_avg.copy()

    speed_after = (
        speed_df[
            (speed_df["New_Measurement"] == incident_measurement)
            & (speed_df["StartTime"] >= acc_start_time)
        ]
        .sort_values("StartTime")
        .reset_index(drop=True)
    )

    speed_after["is_congested"] = speed_after["V_mean"] <= Vc

    T2 = None

    for i in range(len(speed_after) - continuous_n + 1):
        if speed_after.loc[i:i + continuous_n - 1, "is_congested"].all():
            T2 = speed_after.loc[i + continuous_n - 1, "EndTime"]
            break

    if T2 is None:
        dc = np.nan
    else:
        dc = (T2 - acc_start_time) / 60
        if dc == 0:
            dc = np.nan

    print("dc:", dc)

    el = lane - acc_lane

    before_group = f"{acc_start_time - sec_time}~{acc_start_time}"
    after_group = f"{acc_start_time}~{acc_start_time + sec_time}"

    s_before = entry_df.loc[
        (entry_df["TimeGroup"] == before_group)
        & (entry_df["New_Measurement"] == incident_measurement),
        "Phi_진입"
    ].iloc[0]

    hv = heavy_df.loc[
        (heavy_df["TimeGroup"] == after_group)
        & (heavy_df["New_Measurement"] == incident_measurement),
        "rate"
    ].iloc[0]

    stvm_before = stvm_df.loc[
        (stvm_df["TimeGroup"] == before_group)
        & (stvm_df["New_Measurement"] == incident_measurement),
        "STVM"
    ].iloc[0]

    stvm_after = stvm_df.loc[
        (stvm_df["TimeGroup"] == after_group)
        & (stvm_df["New_Measurement"] == incident_measurement),
        "STVM"
    ].iloc[0]

    log_dc_df = pd.DataFrame({
        "Dc": dc,
        "log_Dc": np.log(dc),
        "EL": el,
        "T": (acc_finish_time - acc_start_time) / 60,
        "S_before": s_before,
        "HV": hv,
        "G": lane_gradient,
        "STVM": stvm_before,
        "Delta_STVM": stvm_after - stvm_before,
    }, index=[0])

    return log_dc_df


def centralization_log_d(df):
    df = df.copy()

    means = df[["EL", "STVM", "Delta_STVM", "S_before", "HV", "G"]].mean(numeric_only=True)

    df["EL_c"] = df["EL"] - means["EL"]
    df["I_EL"] = (df["EL"] > 0).astype(int)
    df["T_c"] = df["T"]
    df["S_c"] = df["S_before"] - means["S_before"]
    df["HV_c"] = df["HV"] - means["HV"]
    df["G_c"] = df["G"] - means["G"]
    df["S_c**2"] = df["S_c"] ** 2
    df["STVM_c"] = df["STVM"] - means["STVM"]
    df["Delta_STVM_c"] = df["Delta_STVM"] - means["Delta_STVM"]
    df["EL_c*S_c"] = df["EL_c"] * df["S_c"]
    df["HV_c*G_c"] = df["HV_c"] * df["G_c"]
    df["I_EL_S"] = df["I_EL"] * df["S_c"]

    result_df = df[[
        "Dc", "log_Dc", "EL_c", "I_EL", "T_c", "S_c", "HV_c", "G_c",
        "S_c**2", "STVM_c", "Delta_STVM_c", "EL_c*S_c", "HV_c*G_c", "I_EL_S"
    ]]

    result_df = result_df.round(3)
    return result_df


def save_to_excel(excel_df, folder_path, file_name, color=False):
    result_folder = os.path.join(folder_path, "해상도+모형식 수정")
    os.makedirs(result_folder, exist_ok=True)

    excel_file_name = f"{file_name}.xlsx"
    excel_file_path = os.path.join(result_folder, excel_file_name)

    if color:
        styled = excel_df.style.applymap(excel_color)
        styled.to_excel(excel_file_path, engine="openpyxl")
    else:
        excel_df.to_excel(excel_file_path, index=True)

    print(f"{excel_file_name} 생성 완료")


def excel_color(val):
    if pd.isna(val):
        return ""
    elif val <= 0:
        return "background-color: #FF0000"
    else:
        return "background-color: #FFC000"


# =============================================================================
# 6. 실행부
# =============================================================================
if __name__ == "__main__":
    senario_df = pd.read_csv(senario_path, encoding="cp949")

    tree = ET.parse(inpx_path)
    root = tree.getroot()

    for dcp in root.findall(".//dataCollectionPoint"):
        detector_no = int(dcp.get("no")) % 1000
        detector_link = int(dcp.get("lane").split()[0])
        detector_map[detector_no] = detector_link

    folder_path = path
    parquet_list = sorted([
        file for file in os.listdir(folder_path)
        if file.endswith(".parquet") and not file.startswith("processed_")
    ])

    processed_folder = os.path.join(folder_path, "processed")

    for sec_time_value in sec_time_list:
        sec_time = sec_time_value
        revision_time = 3600 / sec_time

        print(f"\n==============================")
        print(f"집계시간 {sec_time}초 processed 파일 생성/확인")
        print(f"==============================")

        build_processed_files(
            folder_path=folder_path,
            parquet_list=parquet_list,
            processed_folder=processed_folder,
            overwrite=False
        )

        for up_dcp_pos_value in up_dcp_pos_list:
            up_dcp_pos = up_dcp_pos_value

            print(f"\n===== 집계시간 {sec_time}초 / 상류검지기 {up_dcp_pos} =====")

            for continuous_num in continuous_list:
                continuous_n = int(continuous_num / sec_time)

                log_d_df_list = []
                gradient_list = []

                for idx, start in enumerate(range(0, len(parquet_list), num_mer)):
                    print(f"============ idx={idx}, start={start} ============")

                    row = senario_df.iloc[idx]

                    lane = row["lane_count"]
                    acc_lane = row["lane_closure_count"]
                    acc_finish_time = acc_start_time + row["incident_duration_min"] * 60

                    incident_location = row["location_name"]
                    locations = [x.strip() for x in incident_location.split(",")]
                    location = locations[0]

                    incident_measurement, lane_gradient = get_upstream_detector(root, location)
                    gradient_list.append((incident_measurement, lane_gradient))

                    batch_files = parquet_list[start:start + num_mer]

                    print("Vc:", Vc)
                    print(
                        f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] "
                        f"입력값 | 교통량={row['base_main_in_vph']}, "
                        f"기존 차로수={lane}, 유고 차로수={acc_lane}, "
                        f"유고지속시간={row['incident_duration_min']}, "
                        f"유고지점={incident_location}, 검지기={incident_measurement}, "
                        f"lane_gradient={lane_gradient}"
                    )

                    speed_list = []
                    density_list = []
                    heavy_list = []
                    entry_list = []
                    rfr_list_all = []
                    normality_list_all = []
                    stvm_list = []

                    for mer_file in batch_files:
                        processed_name = f"processed_{sec_time}sec_{mer_file}"
                        processed_path = os.path.join(processed_folder, processed_name)

                        print("processed 읽기:", processed_name)

                        processed_df = pd.read_parquet(processed_path)

                        speed_list.append(
                            processed_df[[
                                "StartTime", "EndTime", "TimeGroup", "New_Measurement",
                                "V_mean", "V_count", "delta_V"
                            ]]
                        )

                        density_list.append(
                            processed_df[[
                                "StartTime", "TimeGroup", "New_Measurement", "delta_K"
                            ]]
                        )

                        heavy_list.append(
                            processed_df[[
                                "StartTime", "TimeGroup", "New_Measurement", "rate"
                            ]]
                        )

                        entry_list.append(
                            processed_df[[
                                "StartTime", "TimeGroup", "New_Measurement", "Phi_진입"
                            ]]
                        )

                        rfr_list_all.append(
                            processed_df[[
                                "StartTime", "TimeGroup", "New_Measurement", "RFR"
                            ]]
                        )

                        normality_list_all.append(
                            processed_df[[
                                "StartTime", "TimeGroup", "New_Measurement", "F(outrate)"
                            ]]
                        )

                        stvm_list.append(
                            processed_df[[
                                "StartTime", "EndTime", "TimeGroup", "New_Measurement", "STVM"
                            ]]
                        )

                        del processed_df
                        gc.collect()

                    print("시드 평균 계산 중...")

                    merged_speed = pd.concat(speed_list, ignore_index=True)
                    merged_density = pd.concat(density_list, ignore_index=True)
                    merged_heavy = pd.concat(heavy_list, ignore_index=True)
                    merged_entry = pd.concat(entry_list, ignore_index=True)
                    merged_rfr = pd.concat(rfr_list_all, ignore_index=True)
                    merged_normality = pd.concat(normality_list_all, ignore_index=True)
                    merged_stvm = pd.concat(stvm_list, ignore_index=True)

                    speed_avg = (
                        merged_speed
                        .groupby(["StartTime", "EndTime", "TimeGroup", "New_Measurement"])
                        [["V_mean", "delta_V"]]
                        .mean()
                        .reset_index()
                    )

                    density_avg = (
                        merged_density
                        .groupby(["StartTime", "TimeGroup", "New_Measurement"])
                        [["delta_K"]]
                        .mean()
                        .reset_index()
                    )

                    heavy_avg = (
                        merged_heavy
                        .groupby(["StartTime", "TimeGroup", "New_Measurement"])
                        [["rate"]]
                        .mean()
                        .reset_index()
                    )

                    entry_avg = (
                        merged_entry
                        .groupby(["StartTime", "TimeGroup", "New_Measurement"])
                        [["Phi_진입"]]
                        .mean()
                        .reset_index()
                    )

                    rfr_avg = (
                        merged_rfr
                        .groupby(["StartTime", "TimeGroup", "New_Measurement"])
                        [["RFR"]]
                        .mean()
                        .reset_index()
                    )

                    normality_avg = (
                        merged_normality
                        .groupby(["StartTime", "TimeGroup", "New_Measurement"])
                        [["F(outrate)"]]
                        .mean()
                        .reset_index()
                    )

                    stvm_avg = (
                        merged_stvm
                        .groupby(["TimeGroup", "New_Measurement"], sort=False)
                        .mean(numeric_only=True)
                        .reset_index()
                    )

                    log_d_df = calculate_logD(speed_avg, entry_avg, heavy_avg, stvm_avg)
                    log_d_df_list.append(log_d_df)

                    del speed_list, density_list, heavy_list, entry_list, rfr_list_all, normality_list_all, stvm_list
                    del merged_speed, merged_density, merged_heavy, merged_entry, merged_rfr, merged_normality, merged_stvm
                    del speed_avg, density_avg, heavy_avg, entry_avg, rfr_avg, normality_avg, stvm_avg, log_d_df
                    gc.collect()

                log_d_df_all = pd.concat(log_d_df_list, ignore_index=True)
                final_log_d = centralization_log_d(log_d_df_all)
                final_log_d = final_log_d.fillna("NA")

                save_to_excel(
                    final_log_d,
                    folder_path,
                    f"임계연속시간_영향권5_진출_{sec_time}초_{150 + (up_dcp_pos - 1) * 100}m_테스트"
                )

                del log_d_df_list, log_d_df_all, final_log_d
                gc.collect()



집계시간 10초 processed 파일 생성/확인
processed 생성: processed_10sec_화성~서울(110-영향권5_진출_3차로폐쇄-검증용)_260615_001.parquet
processed 생성: processed_10sec_화성~서울(110-영향권5_진출_3차로폐쇄-검증용)_260615_002.parquet
processed 생성: processed_10sec_화성~서울(110-영향권5_진출_3차로폐쇄-검증용)_260615_003.parquet
processed 생성: processed_10sec_화성~서울(110-영향권5_진출_3차로폐쇄-검증용)_260615_004.parquet
processed 생성: processed_10sec_화성~서울(110-영향권5_진출_3차로폐쇄-검증용)_260615_005.parquet
processed 생성: processed_10sec_화성~서울(110-영향권5_진출_3차로폐쇄-검증용)_260615_006.parquet
processed 생성: processed_10sec_화성~서울(110-영향권5_진출_3차로폐쇄-검증용)_260615_007.parquet
processed 생성: processed_10sec_화성~서울(110-영향권5_진출_3차로폐쇄-검증용)_260615_008.parquet
processed 생성: processed_10sec_화성~서울(110-영향권5_진출_3차로폐쇄-검증용)_260615_009.parquet
processed 생성: processed_10sec_화성~서울(110-영향권5_진출_3차로폐쇄-검증용)_260615_010.parquet
processed 생성: processed_10sec_화성~서울(110-영향권5_진출_3차로폐쇄-검증용)_260615_011.parquet
processed 생성: processed_10sec_화성~서울(110-영향권5_진출_3차로폐쇄-검증용)_260615_012.parquet
processed 생성: processed_10sec_화성~서울